# FINS notebook 11 — a tracer in a hidden flow, the learned arm (exp69; Colab, GPU)

Registered in `README_EXPERIMENTS.md` (exp69, TR1–TR8, 10.10) before any of these runs. Only the passive tracer c of an
f-plane quasi-geostrophic flow is observed; the flow is hidden. This notebook runs the arm that needs a GPU:
* cell 2: three window emulators c_{j+1} = F(c_{j-3}, ..., c_j) (WindowFNO2d, w = 4) at 128^2, seeds 42–44, trained on
  c alone: 1024 port trajectories of data seed 6000 + s (generated on the GPU, about 2–4 min per seed), 40 Fourier
  modes, 10000 steps of 32. Roughly 45–70 min each on a T4 (the first log lines show the speed); each saves
  `<checkpoint>.partial` to Drive every 1000 steps and resumes from it, so a disconnect costs at most 1000 steps: rerun
  cells 1 and 2.
* cell 3: per emulator, its errors, the rotation commutator, and FINS's window pencil on its own rollouts (self) and on
  the data (data). A few minutes each.
* cell 4 shows progress; cell 5 zips the json and logs to `fins_nb11.zip` (small: upload this one to the chat), in
  /content and on My Drive. The emulators (about 100 MB each) stay on Drive under fins_runs/results/tracer.

The CPU arms (the simulator oracle, weak SINDy on c, the best hidden lift) run on our side.

Before running, upload the newest **`fins_repo_full.zip`** from the chat to My Drive (root or one folder down).

In [ ]:
# 1. setup: Drive, the newest repository zip (with exp69), pyqg from GitHub, GPU check
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
os.makedirs(os.path.join(R, 'tracer'), exist_ok=True)


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_exp69(z):
    with zipfile.ZipFile(z) as zz:
        return 'fins_repo/experiments/exp69_hidden_tracer.py' in zz.namelist()


ZIP_PATH = newest('fins_repo_full*.zip', has_exp69)
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive has exp69: upload the newest zip from the chat')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/pyqg/pyqg.git'], check=False)
chk = subprocess.run([sys.executable, '-c', 'import exp69_hidden_tracer'], cwd=EXPDIR, capture_output=True, text=True)
if chk.returncode:
    raise SystemExit('exp69 does not import here:\n' + chk.stderr[-2000:])
chk = subprocess.run([sys.executable, 'exp69_hidden_tracer.py', 'check', '--n', '64'], cwd=EXPDIR, capture_output=True,
                     text=True)
print(chk.stdout[-2500:], chk.stderr[-800:])
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU (or better), then rerun this cell')
CACHE = '/content/tracer_cache'
RUNS = os.path.join(R, 'tracer_runs')
os.makedirs(os.path.join(RUNS, 'logs'), exist_ok=True)
SEEDS = (42, 43, 44)
COMMON = ['--n', '128', '--w', '4', '--cache', CACHE]


def go(stem, cmd, keys):
    if os.path.exists(stem + '.json'):
        print('done', os.path.basename(stem))
        return
    log = os.path.join(RUNS, 'logs', os.path.basename(stem) + '.log')
    t0 = time.time()
    with open(log, 'a') as fh:
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        rc = subprocess.call([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    lines = open(log).read().splitlines()
    keep = [l.strip() for l in lines if any(k in l for k in keys)] if rc == 0 else lines[-6:]
    print(os.path.basename(stem), 'exit', rc, f'{time.time() - t0:.0f} s', *[k[:170] for k in keep[-8:]], sep='\n   ')


def ckpt(s):
    return os.path.join(R, 'tracer', f'wfno_n128_k0.05_w4_s{s}.pt')

In [ ]:
# 2. train the three window emulators (resumable)
for s in SEEDS:
    if os.path.exists(ckpt(s)):
        print('trained', os.path.basename(ckpt(s)))
        continue
    log = os.path.join(RUNS, 'logs', f'train_w4_s{s}.log')
    t0 = time.time()
    print(f's{s}: training (log {log})', flush=True)
    with open(log, 'a') as fh:
        cmd = ['exp69_hidden_tracer.py', 'train', '--seed', str(s), *COMMON, '--modes', '40', '--n-traj', '1024',
               '--steps', '10000', '--batch', '32', '--log-every', '500', '--ckpt-every', '1000']
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        p = subprocess.Popen([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
        seen = len(open(log).read().splitlines())
        while p.poll() is None:
            time.sleep(60)
            lines = open(log).read().splitlines()
            for l in lines[seen:]:
                print('   ', l.strip()[:170], flush=True)
            seen = len(lines)
    lines = open(log).read().splitlines()
    print(f's{s}: exit {p.returncode}, {time.time() - t0:.0f} s', *[l[:170] for l in lines[seen:]], sep='\n   ')
left = [f's{s}' for s in SEEDS if not os.path.exists(ckpt(s))]
print('all three emulators trained' if not left else 'NOT YET: ' + ', '.join(left))

In [ ]:
# 3. FINS through each trained emulator: on its own rollouts (self) and on the data (data)
for s in SEEDS:
    if not os.path.exists(ckpt(s)):
        print('not trained yet: s', s)
        continue
    for arm in ('self', 'data'):
        stem = os.path.join(RUNS, f'ex69_{arm}_fno_w4_s{s}')
        go(stem, ['exp69_hidden_tracer.py', 'fins', '--arm', arm, '--seed', str(s), *COMMON, '--save', stem],
           ('held-out error', 'own rollouts', 'lowest', 'observed:', 'heat-false', 'H (10', 'span6'))

In [ ]:
# 4. progress
nt = sum(os.path.exists(ckpt(s)) for s in SEEDS)
na = len(glob.glob(os.path.join(RUNS, 'ex69_*_fno_w4_*.json')))
print(f'trained {nt}/3; FINS results {na}/6')
for f in sorted(glob.glob(os.path.join(R, 'tracer', '*.partial'))):
    print('in progress:', os.path.basename(f), time.ctime(os.path.getmtime(f)))

In [ ]:
# 5. copy the json and logs file by file to local disk (remount if Drive drops); fins_nb11.zip in /content and on Drive
local = '/content/fins_nb11'
shutil.rmtree(local, ignore_errors=True)
files = glob.glob(os.path.join(RUNS, '*.json')) + glob.glob(os.path.join(RUNS, 'logs', '*.log'))
for f in sorted(set(files)):
    dst = os.path.join(local, os.path.relpath(f, R))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for attempt in range(3):
        try:
            shutil.copy2(f, dst)
            break
        except OSError as e:
            print('retry', f, e)
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)
out = '/content/fins_nb11.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(local, '**', '*'), recursive=True)):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, local))
print(out, os.path.getsize(out), 'bytes')
try:
    shutil.copy2(out, '/content/drive/MyDrive/fins_nb11.zip')
    print('copied to MyDrive: fins_nb11.zip (upload this one to the chat)')
except OSError as e:
    print('Drive copy failed; download /content/fins_nb11.zip from the file pane:', e)